## CNN Model Testing (Izzie Lee)

In [ ]:
import torch
from torchvision import datasets, transforms

# load train and test dataset paths
train_path = '/Users/jiwonii/Desktop/Training'
test_path = '/Users/jiwonii/Desktop/Testing'

# transform for training images with data augmentation (resize, flips, rotation) + normalization
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(),  # 50% chance to flip images horizontally or vertically
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=20),  # randomly rotate by up to 20 degrees
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# transform for validation/testing images: resize + normalize only, no augmentation
eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# load the Training folder twice (once per transform) so the val split gets clean, unaugmented images
train_dataset_aug = datasets.ImageFolder(root=train_path, transform=train_transform)
train_dataset_eval = datasets.ImageFolder(root=train_path, transform=eval_transform)

# carve a validation split out of Training; the Testing folder stays untouched until final evaluation
val_ratio = 0.15
val_size = int(len(train_dataset_aug) * val_ratio)
train_size = len(train_dataset_aug) - val_size

generator = torch.Generator().manual_seed(42)
train_split, val_split = torch.utils.data.random_split(range(len(train_dataset_aug)), [train_size, val_size], generator=generator)

train_dataset = torch.utils.data.Subset(train_dataset_aug, train_split.indices)
val_dataset = torch.utils.data.Subset(train_dataset_eval, val_split.indices)
test_dataset = datasets.ImageFolder(root=test_path, transform=eval_transform)

# class names corresponding to the folder names
class_names = train_dataset_aug.classes

# image data loading in batches
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Data Loaded Successfully... train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")

In [ ]:
import matplotlib.pyplot as plt

# pulls a batch of images to display
images, labels = next(iter(train_loader))

# plot the first 10 images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for i in range(10):
    # undo the (mean=0.5, std=0.5) normalization, then CHW -> HWC for matplotlib
    img = images[i] * 0.5 + 0.5
    img = img.permute(1, 2, 0).numpy()

    axes[i].imshow(img)
    axes[i].set_title(class_names[labels[i]])
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class ImprovedCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(ImprovedCNN, self).__init__()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)  # cut image size by half
        self.spatial_dropout = nn.Dropout2d(0.1)  # light regularization on feature maps between conv blocks

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)  # input size: 256x256
        self.bn1 = nn.BatchNorm2d(num_features=16)

        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)  # input size: 128x128
        self.bn2 = nn.BatchNorm2d(num_features=32)

        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)  # input size: 64x64
        self.bn3 = nn.BatchNorm2d(num_features=64)

        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)  # input size: 32x32
        self.bn4 = nn.BatchNorm2d(num_features=128)

        self.global_pool = nn.AdaptiveAvgPool2d(1)  # 128 x 16 x 16 -> 128 x 1 x 1

        self.dropout = nn.Dropout(0.5)
        self.full_conn1 = nn.Linear(in_features=128, out_features=128)
        self.full_conn2 = nn.Linear(in_features=128, out_features=num_classes)

    def forward(self, x):
        x = self.spatial_dropout(self.pool(F.relu(self.bn1(self.conv1(x)))))  # output size: 128x128
        x = self.spatial_dropout(self.pool(F.relu(self.bn2(self.conv2(x)))))  # output size: 64x64
        x = self.spatial_dropout(self.pool(F.relu(self.bn3(self.conv3(x)))))  # output size: 32x32
        x = self.spatial_dropout(self.pool(F.relu(self.bn4(self.conv4(x)))))  # output size: 16x16

        x = self.global_pool(x)
        x = torch.flatten(x, start_dim=1)  # (batch, 128) instead of (batch, 128*16*16)

        x = F.relu(self.full_conn1(x))
        x = self.dropout(x)
        x = self.full_conn2(x)
        return x

model = ImprovedCNN()
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {total_params:,}")

In [ ]:
import copy
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

device = torch.device("cpu")
print(f"Using device: {device}")

# move the model to the chosen device (cpu)
model = model.to(device)

# define loss function (criterion), optimizer, and adaptive lr scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

# set num of epochs (passes through the dataset) and early stopping patience
epochs = 40
early_stop_patience = 6

# init lists to track history of stats
train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

best_val_acc = 0.0
best_model_state = None
epochs_without_improvement = 0

for epoch in range(epochs):
    # ------ TRAINING PHASE ------
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)  # _ represents the max logit value; we only care about predicted index value of logit
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_train_loss = running_loss / total_train
    epoch_train_acc = (correct_train / total_train) * 100

    # -------- VALIDATION PHASE --------
    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():  # disable gradient calculation for efficiency, since no longer needed outside training
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = (correct_val / total_val) * 100

    # step scheduler on validation accuracy, keeping the test set fully untouched during training
    scheduler.step(epoch_val_acc)

    # save epoch stats to history lists
    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_acc)
    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_acc)

    print(f"Epoch {epoch+1}/{epochs} "
          f"| Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}% "
          f"| Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}%")

    # -------- EARLY STOPPING / CHECKPOINTING --------
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= early_stop_patience:
            print(f"No validation improvement for {early_stop_patience} epochs, stopping early at epoch {epoch+1}.")
            break

# restore the best-performing weights (by validation accuracy) before final evaluation
model.load_state_dict(best_model_state)
print(f"Training finished! Best validation accuracy: {best_val_acc:.4f}%")

In [ ]:
import matplotlib.pyplot as plt
epochs_range = range(1, len(train_losses) + 1)

# create a layout for plots
fig, (plot1, plot2) = plt.subplots(1, 2, figsize=(14, 5))

# plot 1: training vs validation loss
plot1.plot(epochs_range, train_losses, label='Training Loss', color='red', marker='o')
plot1.plot(epochs_range, val_losses, label='Validation Loss', color='purple', marker='o')
plot1.set_title('Loss over Epochs')
plot1.set_xlabel('Epochs')
plot1.set_ylabel('Loss')
plot1.grid(True, linestyle='--', alpha=0.6)
plot1.legend()

# plot 2: training vs validation accuracy
plot2.plot(epochs_range, train_accuracies, label='Train Accuracy', color='blue', marker='o')
plot2.plot(epochs_range, val_accuracies, label='Validation Accuracy', color='green', marker='o')
plot2.set_title('Accuracy over Epochs')
plot2.set_xlabel('Epochs')
plot2.set_ylabel('Accuracy (%)')
plot2.grid(True, linestyle='--', alpha=0.6)
plot2.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = (np.array(all_preds) == np.array(all_labels)).mean() * 100
print(f"Final Held-Out Test Accuracy: {test_acc:.4f}%\n")

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

# confusion matrix heatmap
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix (Held-Out Test Set)')
plt.tight_layout()
plt.show()